In [15]:
import re
from html.parser import HTMLParser
from urllib.parse import unquote, urljoin
from urllib.request import Request, urlopen

TRANSCRIPT_FILE_PATTERN = re.compile(
    r"(detected[_\-. ]*transcripts?|transcripts?|molecules?|(^|[_\-.])tx([_\-.]|$))",
    re.IGNORECASE,
)


class LinkParser(HTMLParser):
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag.lower() != "a":
            return
        href = dict(attrs).get("href")
        if href:
            self.links.append(href)


def normalize_gse(value):
    match = re.search(r"\bGSE\d+\b", str(value).upper())
    return match.group(0) if match else None


def geo_suppl_url(gse):
    gse = normalize_gse(gse)
    if not gse:
        return None
    digits = gse[3:]
    family = f"GSE{digits[:-3]}nnn"
    return f"https://ftp.ncbi.nlm.nih.gov/geo/series/{family}/{gse}/suppl/"


def read_url(url, timeout=30):
    request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urlopen(request, timeout=timeout) as response:
        return response.read().decode("utf-8", errors="replace")


def parse_suppl_links(html):
    parser = LinkParser()
    parser.feed(html)
    names = []
    for href in parser.links:
        name = unquote(href.rstrip("/").split("/")[-1])
        if name and name not in {".", ".."} and not name.lower().endswith("filelist.txt"):
            names.append((name, href))
    return names


def records_from_filelist(gse, filelist_url):
    filelist = pd.read_csv(filelist_url, sep="\t")
    name_col = "Name" if "Name" in filelist.columns else filelist.columns[0]
    return [
        {"gse": gse, "name": str(name), "url": urljoin(filelist_url, str(name)), "source": "filelist.txt"}
        for name in filelist[name_col].dropna().astype(str).unique()
    ]


def records_from_suppl_index(gse, suppl_url):
    html = read_url(suppl_url)
    return [
        {"gse": gse, "name": name, "url": urljoin(suppl_url, href), "source": "/suppl/ index"}
        for name, href in parse_suppl_links(html)
    ]


def fetch_geo_suppl_files(gse, prefer_filelist=True):
    gse = normalize_gse(gse)
    suppl_url = geo_suppl_url(gse)
    errors = []
    if not gse or not suppl_url:
        return [], ["Not a GEO Series accession"]

    if prefer_filelist:
        filelist_url = urljoin(suppl_url, "filelist.txt")
        try:
            records = records_from_filelist(gse, filelist_url)
            if records:
                return records, errors
            errors.append("filelist.txt was empty")
        except Exception as exc:
            errors.append(f"filelist.txt unavailable: {exc}")

    try:
        return records_from_suppl_index(gse, suppl_url), errors
    except Exception as exc:
        errors.append(f"/suppl/ index unavailable: {exc}")
        return [], errors


def is_transcript_level_file(name):
    return bool(TRANSCRIPT_FILE_PATTERN.search(str(name)))

In [16]:
import pandas as pd

df = pd.read_csv('museum_of_st_meta.csv')
# df = df[~df['accession'].isna()&(df['dataset_title'].str.contains('Visium'))]
df['dataset_type'].unique().shape

(312,)

In [17]:
image_based_transcriptomics = [
    "CosMX",
    "Xenium",
    "MERFISH",
    "seqFISH",
    "seqFISH+",
    "osmFISH",
    "smFISH",
    "bDNA-smFISH",
    "bDNA-MERFISH",
    "HCR-seqFISH",
    "ExM-MERFISH",
    "split-FISH",
    "par-seqFISH",
    "corrFISH",
    "SRM seqFISH",
    "iceFISH",
    "C-FISH",
    "RNAscope",
    "Molecular Cartography",
    "coppaFISH",
    "HybISS",
    "HybRISS",
    "ISS",
    "IISS",
    "FISSEQ",
    "STARmap",
    "STARmap PLUS",
    "Deep-STARmap",
    "ExSeq",
    "EASI-FISH",
    "SCRINSHOT",
    "CISI",
    "MOSAICA",
    "PHYTOMap",
    "ARTseq-FISH",
    "clampFISH 2.0",
    "mFISH3D",
    "Spectrum-FISH",
    "USeqFISH",
    "FISHnCHIPs",
    "DART-FISH",
    "Perturb-FISH",
    "cycleHCR",
    "bacterial-MERFISH",
    "weMERFISH",
    "hamFISH",
    "3DEEP-FISH",
    "HiPlex FISH",
    "ATLAS",
    "RCA-MERFISH",
    "PRISM",
    "RAM-FISH",
    "GenePS",
    "RAEFISH",
    "BCR-MERFISH",
    "split-roll FISH",
    "TEMPOmap",
    "RNAsky",
    "PerturbView",
    "SWITCH-seq",
    "combFISH",
    "Cartana",
]

In [18]:

IMAGE_BASED_FILTER_COLUMNS = ['dataset_type', 'assays']


def image_based_keyword_matches(row):
    text = ' | '.join(row[col] for col in IMAGE_BASED_FILTER_COLUMNS if pd.notna(row[col]))
    matches = []
    for keyword in image_based_transcriptomics:
        pattern = rf"(?<![A-Za-z0-9]){re.escape(keyword)}(?![A-Za-z0-9])"
        if re.search(pattern, text, flags=re.IGNORECASE):
            matches.append(keyword)
    return '; '.join(matches)


image_based_subset = df.copy()
image_based_subset['image_based_keyword_matches'] = image_based_subset.apply(image_based_keyword_matches, axis=1)
image_based_subset = image_based_subset[image_based_subset['image_based_keyword_matches'] != ''].reset_index(drop=True)
image_based_subset.shape


(937, 36)

In [19]:

accession_cols = ['accession', 'data_link']
accessions = (
    image_based_subset[accession_cols]
    .stack()
    .dropna()
    .astype(str)
    .str.strip()
)
accessions = sorted({value for value in accessions if value and value.lower() != 'nan'})
len(accessions)


84

In [20]:

gse_ids = sorted({gse for gse in (normalize_gse(a) for a in accessions) if gse})
print(len(gse_ids))
print(gse_ids)


29
['GSE199463', 'GSE237086', 'GSE253439', 'GSE260452', 'GSE263793', 'GSE274042', 'GSE275954', 'GSE277463', 'GSE277782', 'GSE277936', 'GSE278973', 'GSE282059', 'GSE283625', 'GSE283832', 'GSE286964', 'GSE287648', 'GSE289263', 'GSE290577', 'GSE290835', 'GSE290921', 'GSE292394', 'GSE293124', 'GSE294965', 'GSE297091', 'GSE297945', 'GSE298602', 'GSE299434', 'GSE300303', 'GSE304265']


In [21]:

print(gse_ids[-20:])


['GSE277936', 'GSE278973', 'GSE282059', 'GSE283625', 'GSE283832', 'GSE286964', 'GSE287648', 'GSE289263', 'GSE290577', 'GSE290835', 'GSE290921', 'GSE292394', 'GSE293124', 'GSE294965', 'GSE297091', 'GSE297945', 'GSE298602', 'GSE299434', 'GSE300303', 'GSE304265']


In [22]:

geo_files = []
fetch_errors = {}
missing_gses = []

for gse in gse_ids:
    records, errors = fetch_geo_suppl_files(gse)
    if records:
        geo_files.extend(records)
    else:
        missing_gses.append(gse)
    if errors:
        fetch_errors[gse] = errors

geo_files_df = pd.DataFrame(geo_files)
geo_files_df.head()


,gse,name,url,source
0,GSE199463,GSE199463_RAW.tar,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE199...,filelist.txt
1,GSE199463,GSM5973653_32783-slide1_A1_DAPI.tiff.gz,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE199...,filelist.txt
2,GSE199463,GSM5973653_32783-slide1_A1_DAPI.tiff_counts_me...,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE199...,filelist.txt
3,GSE199463,GSM5973653_32783-slide1_A1_raw.tiff.gz,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE199...,filelist.txt
4,GSE199463,GSM5973653_32783-slide1_A1_results.txt.gz,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE199...,filelist.txt


In [23]:

fetch_errors


{'GSE277782': ['filelist.txt unavailable: HTTP Error 404: Not Found'],
 'GSE277936': ['filelist.txt unavailable: HTTP Error 404: Not Found'],
 'GSE282059': ['filelist.txt unavailable: HTTP Error 404: Not Found'],
 'GSE283625': ['filelist.txt unavailable: HTTP Error 404: Not Found'],
 'GSE287648': ['filelist.txt unavailable: HTTP Error 404: Not Found'],
 'GSE290921': ['filelist.txt unavailable: HTTP Error 404: Not Found'],
 'GSE304265': ['filelist.txt unavailable: HTTP Error 404: Not Found']}

In [24]:

fetch_summary = pd.DataFrame(
    {
        "total_gses": [len(gse_ids)],
        "gses_with_files": [geo_files_df["gse"].nunique() if not geo_files_df.empty else 0],
        "missing_gses": [len(missing_gses)],
        "filelist_or_suppl_errors": [len(fetch_errors)],
    }
)

fetch_summary


,total_gses,gses_with_files,missing_gses,filelist_or_suppl_errors
0,29,29,0,7


In [25]:

if geo_files_df.empty:
    tx_matches = pd.DataFrame(columns=["gse", "name", "url", "source"])
else:
    tx_matches = geo_files_df[geo_files_df["name"].map(is_transcript_level_file)].copy()
    tx_matches = tx_matches.sort_values(["gse", "name"]).reset_index(drop=True)

tx_gses = sorted(tx_matches["gse"].unique())
tx_avail = len(tx_gses)

print(f"GSEs with transcript-level files: {tx_avail} / {len(gse_ids)}")
tx_matches[["gse", "name", "source", "url"]]


GSEs with transcript-level files: 19 / 29


,gse,name,source,url
0,GSE237086,GSM7594569_exp1_detected_transcripts.csv.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237...
1,GSE237086,GSM7594570_exp2_detected_transcripts.csv.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237...
2,GSE237086,GSM7594571_exp3_detected_transcripts.csv.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237...
3,GSE237086,GSM7594572_exp4_detected_transcripts.csv.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237...
4,GSE237086,GSM7594573_WT_detected_transcripts.csv.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE237...
...,...,...,...,...
170,GSE300303,GSM9058064_TR-2-24B_transcripts.parquet.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE300...
171,GSE300303,GSM9058065_TR-2-24C_transcripts.parquet.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE300...
172,GSE300303,GSM9058066_TR-2-24D_transcripts.parquet.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE300...
173,GSE300303,GSM9058067_TR-2-24E_transcripts.parquet.gz,filelist.txt,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE300...


In [26]:

tx_matches['gse'].drop_duplicates().reset_index(drop=True)


0     GSE237086
1     GSE253439
2     GSE260452
3     GSE263793
4     GSE274042
5     GSE275954
6     GSE277463
7     GSE277782
8     GSE282059
9     GSE283832
10    GSE286964
11    GSE289263
12    GSE292394
13    GSE293124
14    GSE294965
15    GSE297091
16    GSE297945
17    GSE299434
18    GSE300303
Name: gse, dtype: object

In [27]:

def gses_from_row(row):
    values = [row.get('accession'), row.get('data_link')]
    return sorted({gse for gse in (normalize_gse(value) for value in values) if gse})


def join_values(values):
    values = [str(value) for value in values if pd.notna(value) and str(value)]
    return '; '.join(values)


def transcript_status_for_row(row):
    row_gses = gses_from_row(row)
    if not row_gses:
        return pd.Series(
            {
                'transcript_check_gses': '',
                'transcript_check_error': False,
                'transcript_check_error_detail': '',
                'transcript_file_available': False,
                'transcript_file_names': '',
                'transcript_file_urls': '',
            }
        )

    matches = tx_matches[tx_matches['gse'].isin(row_gses)]
    errors = [f"{gse}: {' | '.join(fetch_errors[gse])}" for gse in row_gses if gse in fetch_errors]

    return pd.Series(
        {
            'transcript_check_gses': '; '.join(row_gses),
            'transcript_check_error': bool(errors),
            'transcript_check_error_detail': '; '.join(errors),
            'transcript_file_available': not matches.empty,
            'transcript_file_names': join_values(matches['name'].drop_duplicates()) if not matches.empty else '',
            'transcript_file_urls': join_values(matches['url'].drop_duplicates()) if not matches.empty else '',
        }
    )


transcript_labels = image_based_subset.apply(transcript_status_for_row, axis=1)
image_based_transcript_availability = pd.concat([image_based_subset, transcript_labels], axis=1)
image_based_transcript_availability.to_csv('image_based_transcript_availability.csv', index=False)

image_based_transcript_availability[
    [
        'dataset_id',
        'dataset_title',
        'dataset_type',
        'accession',
        'data_link',
        'transcript_check_gses',
        'transcript_check_error',
        'transcript_file_available',
        'transcript_file_names',
    ]
].head()


,dataset_id,dataset_title,dataset_type,accession,data_link,transcript_check_gses,transcript_check_error,transcript_file_available,transcript_file_names
0,museum_st_a70a5d8be5d9,Same-Slide Spatial Multi-Omics Integration Rev...,CosMX,NaN,NaN,,False,False,
1,museum_st_c7c92aa38dbc,Same-Slide Spatial Multi-Omics Integration Rev...,CosMX,NaN,NaN,,False,False,
2,museum_st_dd8f690a17b0,Single cell systems biology by super-resolutio...,SRM seqFISH,NaN,NaN,,False,False,
3,museum_st_ce748787c7f2,Single-chromosome transcriptional profiling re...,iceFISH,NaN,NaN,,False,False,
4,museum_st_b3008be9dcea,Image-based transcriptomics in thousands of si...,bDNA-smFISH,NaN,NaN,,False,False,


In [29]:
image_based_transcript_availability['transcript_check_gses'].unique()

array(['', 'GSE199463', 'GSE263793', 'GSE260452', 'GSE283625',
       'GSE286964', 'GSE287648', 'GSE290577', 'GSE290921', 'GSE290835',
       'GSE278973', 'GSE277782', 'GSE277463', 'GSE237086', 'GSE294965',
       'GSE282059', 'GSE253439', 'GSE275954', 'GSE292394', 'GSE298602',
       'GSE297945', 'GSE277936', 'GSE297091', 'GSE299434', 'GSE274042',
       'GSE300303', 'GSE289263', 'GSE283832', 'GSE304265', 'GSE293124'],
      dtype=object)

In [31]:
image_based_transcript_availability[image_based_transcript_availability['transcript_check_error']==True]

,dataset_id,source_database,collection_id,collection_name,dataset_title,dataset_type,assays,tissues,tissue_general,diseases,...,n_cells_source,dataset_total_cell_count_source,median_n_measured_vars_source,image_based_keyword_matches,transcript_check_gses,transcript_check_error,transcript_check_error_detail,transcript_file_available,transcript_file_names,transcript_file_urls
543,museum_st_ba2db062abdb,museum_of_spatial_transcriptomics,40571096.0,Histopathologic Evaluation and Single-Cell Spa...,Histopathologic Evaluation and Single-Cell Spa...,CosMX,['CosMX'],"['colon (IPAA pouch non-failure)', 'colon']","['colon', 'colon (IPAA pouch non-failure)']",['pathological'],...,NaN,NaN,museum.n_genes,CosMX,GSE283625,True,GSE283625: filelist.txt unavailable: HTTP Erro...,False,,
544,museum_st_c9a4a72b2fe1,museum_of_spatial_transcriptomics,40571096.0,Histopathologic Evaluation and Single-Cell Spa...,Histopathologic Evaluation and Single-Cell Spa...,CosMX,['CosMX'],"['colon (IPAA pouch failure)', 'colon']","['colon', 'colon (IPAA pouch failure)']",['pathological'],...,NaN,NaN,museum.n_genes,CosMX,GSE283625,True,GSE283625: filelist.txt unavailable: HTTP Erro...,False,,
560,museum_st_52652378fc58,museum_of_spatial_transcriptomics,NaN,comBO: A combined human bone and lympho-myeloi...,comBO: A combined human bone and lympho-myeloi...,Xenium,['Xenium'],"['bone marrow trephines (multiple myeloma)', '...","['bone', 'bone marrow trephines (multiple myel...",['pathological'],...,NaN,NaN,museum.n_genes,Xenium,GSE287648,True,GSE287648: filelist.txt unavailable: HTTP Erro...,False,,
615,museum_st_a0d02e5515b4,museum_of_spatial_transcriptomics,NaN,CD4+ tissue-resident memory Th17 cells are a m...,CD4+ tissue-resident memory Th17 cells are a m...,CosMX,['CosMX'],"['Spondyloarthritis synovium', 'bone']","['bone', 'Spondyloarthritis synovium']",['pathological'],...,NaN,NaN,museum.n_genes,CosMX,GSE290921,True,GSE290921: filelist.txt unavailable: HTTP Erro...,False,,
650,museum_st_53a104e8ec99,museum_of_spatial_transcriptomics,40269162.0,Spatial mapping of transcriptomic plasticity i...,Spatial mapping of transcriptomic plasticity i...,CosMX,['CosMX'],"['PDAC', 'pancreas']","['pancreas', 'PDAC']",['pathological'],...,NaN,NaN,museum.n_genes,CosMX,GSE277782,True,GSE277782: filelist.txt unavailable: HTTP Erro...,True,GSE277782_Run5961_PDAC_Slide1_tx_file.csv.gz; ...,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE277...
698,museum_st_20125b723b1f,museum_of_spatial_transcriptomics,40399382.0,Multiomic analysis of human kidney disease ide...,Multiomic analysis of human kidney disease ide...,CosMX,['CosMX'],"['unilateral ureteric obstruction', 'kidney']","['kidney', 'unilateral ureteric obstruction']",['pathological'],...,NaN,NaN,museum.n_genes,CosMX,GSE282059,True,GSE282059: filelist.txt unavailable: HTTP Erro...,True,GSE282059_2B_FFPE_tx_file.csv.gz,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE282...
728,museum_st_22b799f6d7df,museum_of_spatial_transcriptomics,40502298.0,Enhanced Spatial Transcriptomics Analysis of M...,Enhanced Spatial Transcriptomics Analysis of M...,Xenium,['Xenium'],['lung'],['lung'],['normal'],...,NaN,NaN,museum.n_genes,Xenium,GSE277936,True,GSE277936: filelist.txt unavailable: HTTP Erro...,False,,
729,museum_st_44833a30ac8b,museum_of_spatial_transcriptomics,40502298.0,Enhanced Spatial Transcriptomics Analysis of M...,Enhanced Spatial Transcriptomics Analysis of M...,Xenium,['Xenium'],['lung'],['lung'],['pathological'],...,NaN,NaN,museum.n_genes,Xenium,GSE277936,True,GSE277936: filelist.txt unavailable: HTTP Erro...,False,,
826,museum_st_28915e4a81b6,museum_of_spatial_transcriptomics,GSE304265,Targeting CyclinD1-CDK6 to Mitigate Senescence...,Targeting CyclinD1-CDK6 to Mitigate Senescence...,CosMX,['CosMX'],['liver'],['liver'],['normal'],...,NaN,NaN,NaN,CosMX,GSE304265,True,GSE304265: filelist.txt unavailable: HTTP Erro...,False,,


In [ ]:
image_based_transcript_availability

,dataset_id,source_database,collection_id,collection_name,dataset_title,dataset_type,assays,tissues,tissue_general,diseases,...,n_cells_source,dataset_total_cell_count_source,median_n_measured_vars_source,image_based_keyword_matches,transcript_check_gses,transcript_check_error,transcript_check_error_detail,transcript_file_available,transcript_file_names,transcript_file_urls
0,museum_st_a70a5d8be5d9,museum_of_spatial_transcriptomics,39764057.0,Same-Slide Spatial Multi-Omics Integration Rev...,Same-Slide Spatial Multi-Omics Integration Rev...,CosMX,['CosMX'],"['diffuse large B-cell lymphoma EBV+', 'lymph_...","['lymph_node', 'diffuse large B-cell lymphoma ...",['pathological'],...,NaN,NaN,NaN,CosMX,,False,,False,,
1,museum_st_c7c92aa38dbc,museum_of_spatial_transcriptomics,39764057.0,Same-Slide Spatial Multi-Omics Integration Rev...,Same-Slide Spatial Multi-Omics Integration Rev...,CosMX,['CosMX'],"['diffuse large B-cell lymphoma EBV-', 'lymph_...","['lymph_node', 'diffuse large B-cell lymphoma ...",['pathological'],...,NaN,NaN,NaN,CosMX,,False,,False,,
2,museum_st_dd8f690a17b0,museum_of_spatial_transcriptomics,22660740.0,Single cell systems biology by super-resolutio...,Single cell systems biology by super-resolutio...,SRM seqFISH,['SRM seqFISH'],[],[],['normal'],...,museum.n_cells/bins/spots,museum.n_cells/bins/spots,museum.n_genes,seqFISH; SRM seqFISH,,False,,False,,
3,museum_st_ce748787c7f2,museum_of_spatial_transcriptomics,23416756.0,Single-chromosome transcriptional profiling re...,Single-chromosome transcriptional profiling re...,iceFISH,['iceFISH'],['HeLa'],['HeLa'],['normal'],...,NaN,NaN,museum.n_genes,iceFISH,,False,,False,,
4,museum_st_b3008be9dcea,museum_of_spatial_transcriptomics,24097269.0,Image-based transcriptomics in thousands of si...,Image-based transcriptomics in thousands of si...,bDNA-smFISH,['bDNA-smFISH'],['HeLa'],['HeLa'],['normal'],...,museum.n_cells/bins/spots,museum.n_cells/bins/spots,museum.n_genes,smFISH; bDNA-smFISH,,False,,False,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
932,museum_st_d8e07523223d,museum_of_spatial_transcriptomics,40205045.0,Spatial multi-omics reveals cell-type-specific...,Spatial multi-omics reveals cell-type-specific...,DNA seqFISH+,['DNA seqFISH+'],['mESC'],['mESC'],['normal'],...,NaN,NaN,NaN,seqFISH; seqFISH+,,False,,False,,
933,museum_st_4b6050b667de,museum_of_spatial_transcriptomics,40205045.0,Spatial multi-omics reveals cell-type-specific...,Spatial multi-omics reveals cell-type-specific...,DNA seqFISH+,['DNA seqFISH+'],"['cerebellum', 'brain']","['brain', 'cerebellum']",['normal'],...,NaN,NaN,NaN,seqFISH; seqFISH+,,False,,False,,
934,museum_st_73a0a1006cd7,museum_of_spatial_transcriptomics,NaN,Single-cell multiomics identifies both shared ...,Single-cell multiomics identifies both shared ...,CosMX,['CosMX'],"['colon (Parkinson’s disease)', 'colon']","['colon', 'colon (Parkinson’s disease)']",['pathological'],...,NaN,NaN,NaN,CosMX,,False,,False,,
935,museum_st_66ac6cee9693,museum_of_spatial_transcriptomics,NaN,The Spatial Atlas of Human Anatomy (SAHA): A M...,The Spatial Atlas of Human Anatomy (SAHA): A M...,CosMX,['CosMX'],[],[],['normal'],...,NaN,NaN,NaN,CosMX,,False,,False,,


In [ ]:

# GOOGLE_SHEET_NAME = 'image_based_transcript_availability'
# GOOGLE_WORKSHEET_NAME = 'museum_of_st_meta'

# try:
#     import gspread
#     from gspread_dataframe import set_with_dataframe
# except ImportError:
#     print(
#         "Install gspread and gspread-dataframe to export to Google Sheets, "
#         "or use image_based_transcript_availability.csv."
#     )
# else:
#     try:
#         gc = gspread.service_account()
#         try:
#             spreadsheet = gc.open(GOOGLE_SHEET_NAME)
#         except gspread.SpreadsheetNotFound:
#             spreadsheet = gc.create(GOOGLE_SHEET_NAME)

#         try:
#             worksheet = spreadsheet.worksheet(GOOGLE_WORKSHEET_NAME)
#             worksheet.clear()
#         except gspread.WorksheetNotFound:
#             worksheet = spreadsheet.add_worksheet(
#                 title=GOOGLE_WORKSHEET_NAME,
#                 rows=max(len(image_based_transcript_availability) + 1, 100),
#                 cols=len(image_based_transcript_availability.columns),
#             )

#         set_with_dataframe(worksheet, image_based_transcript_availability, include_index=False, resize=True)
#         print(spreadsheet.url)
#     except Exception as exc:
#         print(f"Google Sheets export failed: {exc}")
#         print("The same dataframe was saved to image_based_transcript_availability.csv.")


Install gspread and gspread-dataframe to export to Google Sheets, or use image_based_transcript_availability.csv.
